# REBUS belief revision: Conditions 3, 4, and 6

This notebook presents the 11 matched-seed comparisons behind the paper's experiment. **Arm 3 = Condition 3, arm 4 = Condition 4, and arm 6 = Condition 6.** The baseline is Condition 1. All three conditions use the same 15 fixed evaluation scenarios after 50 training episodes per seed.

Condition 3 changes learning rates and action control; Condition 4 changes prior precision and belief prediction-error gain; Condition 6 combines Condition 4's belief settings with Condition 3's action-control settings while retaining the baseline learning rates. The notebook reports the paper's primary `BeliefUpdateScore`, the safety and belief/action measures, per-seed selection decisions, and task outcomes. Condition 6's score analysis is calculated from the saved logs and extends the paper's headline score tables.

**Modal setup:** Upload this `.ipynb` alone in [Modal Notebooks](https://modal.com/notebooks), then run every cell in order. The result logs are embedded in a collapsed data cell below. The default Modal notebook image includes Matplotlib; the analysis uses only Python's standard library plus Matplotlib and IPython display. If the archive is elsewhere, set `ARCHIVE_PATH` in the first code cell. The embedded data contains the results, not the model checkpoints. A local checkout with `logs/` also works.


In [ ]:
EMBEDDED_ARCHIVE_B64 = (
    '...'
)


In [ ]:
from pathlib import Path
import base64, csv, io, json, math, statistics, zipfile
from collections import Counter
from IPython.display import Markdown, display
import matplotlib.pyplot as plt

# Optional overrides; leave as None for automatic discovery.
LOGS_DIR = None       # Example: Path('/mnt/my-volume/logs')
ARCHIVE_PATH = None   # Example: Path('/root/conditions_3_4_6_logs.zip')
SEEDS = (7, 11, 17, 23, 29, 31, 37, 41, 43, 47, 53)
ARMS = (3, 4, 6)
CONTRACT = 'evidence-response-v3'


def find_logs():
    roots = [Path.cwd(), Path('/root'), Path('/mnt')]
    if LOGS_DIR is not None:
        path = Path(LOGS_DIR).expanduser()
        if not path.is_dir():
            raise FileNotFoundError(f'LOGS_DIR does not exist: {path}')
        return path
    for root in roots:
        for path in (root / 'logs', root / 'rebus_analysis_data' / 'logs'):
            if (path / 'seed7-arm3' / 'candidate' / 'metrics.json').is_file():
                return path
    candidates = ([Path(ARCHIVE_PATH).expanduser()] if ARCHIVE_PATH is not None else
                  [root / 'conditions_3_4_6_logs.zip' for root in roots] +
                  [root / 'analysis' / 'conditions_3_4_6_logs.zip' for root in roots])
    for path in candidates:
        if path.is_file():
            destination = Path.cwd() / 'rebus_analysis_data'
            destination.mkdir(exist_ok=True)
            with zipfile.ZipFile(path) as z:
                bad = [name for name in z.namelist()
                       if Path(name).is_absolute() or '..' in Path(name).parts]
                if bad:
                    raise ValueError(f'Unsafe path in archive: {bad[0]}')
                z.extractall(destination)
            logs = destination / 'logs'
            if (logs / 'seed7-arm3' / 'candidate' / 'metrics.json').is_file():
                return logs
            raise FileNotFoundError(f'Archive {path} has no expected logs/ layout')
    if 'EMBEDDED_ARCHIVE_B64' in globals():
        destination = Path.cwd() / 'rebus_analysis_data'
        destination.mkdir(exist_ok=True)
        payload = base64.b64decode(EMBEDDED_ARCHIVE_B64)
        with zipfile.ZipFile(io.BytesIO(payload)) as z:
            bad = [name for name in z.namelist()
                   if Path(name).is_absolute() or '..' in Path(name).parts]
            if bad:
                raise ValueError(f'Unsafe path in embedded archive: {bad[0]}')
            z.extractall(destination)
        logs = destination / 'logs'
        if (logs / 'seed7-arm3' / 'candidate' / 'metrics.json').is_file():
            return logs
        raise FileNotFoundError('Embedded archive has no expected logs/ layout')
    raise FileNotFoundError('Set LOGS_DIR or ARCHIVE_PATH to the uploaded results.')

logs = find_logs()
print('Reading results from:', logs.resolve())


## 1. Load and validate the matched runs

Only the current `metrics.json`, `trace.json`, and `selection.json` are used. Archived `pre-*` snapshots use earlier evaluation rules and are excluded. The baseline for a given seed is repeated inside each arm directory; it is checked for equality and counted once in comparisons.


In [ ]:
METRICS = ('BeliefUpdateScore', 'revision_accuracy', 'novel_reversal_revision_accuracy',
           'ignored_evidence_rate', 'moved_but_uncertain_rate', 'entropy_mean',
           'perseveration', 'unsafe_commit_rate')
runs = {}
traces = {}
selections = {}
for arm in ARMS:
    for seed in SEEDS:
        folder = logs / f'seed{seed}-arm{arm}'
        if not folder.is_dir():
            raise FileNotFoundError(folder)
        pair = {}
        for role in ('baseline', 'candidate'):
            metrics_path = folder / role / 'metrics.json'
            trace_path = folder / role / 'trace.json'
            pair[role] = json.loads(metrics_path.read_text())
            traces[arm, seed, role] = json.loads(trace_path.read_text())
            m = pair[role]
            assert m['seed'] == seed and m['episodes'] == 50 and m['n_scenarios'] == 15, metrics_path
            assert m['evaluation_contract_version'] == CONTRACT, metrics_path
            assert len(traces[arm, seed, role]) == 15, trace_path
            assert all(k in m for k in METRICS), metrics_path
        runs[arm, seed] = pair
        selections[arm, seed] = json.loads((folder / 'selection' / 'selection.json').read_text())
        assert selections[arm, seed]['seed'] == seed
        assert selections[arm, seed]['baseline_evaluation_contract_version'] == CONTRACT
        assert selections[arm, seed]['candidate_evaluation_contract_version'] == CONTRACT
        assert math.isclose(selections[arm, seed]['score_gain'],
                            pair['candidate']['BeliefUpdateScore'] - pair['baseline']['BeliefUpdateScore'],
                            abs_tol=1e-10)

for seed in SEEDS:
    reference = runs[3, seed]['baseline']
    for arm in (4, 6):
        assert runs[arm, seed]['baseline'] == reference, f'Baseline mismatch for seed {seed}'

expected_constants = {
    3: dict(prior_precision=1.5, pe_gain=0.35, reward_lr=0.30, punishment_lr=0.22, stickiness=0.25),
    4: dict(prior_precision=0.5, pe_gain=1.0, reward_lr=0.15, punishment_lr=0.15, stickiness=0.75),
    6: dict(prior_precision=0.5, pe_gain=1.0, reward_lr=0.15, punishment_lr=0.15, stickiness=0.25),
}
for arm in ARMS:
    for seed in SEEDS:
        actual = runs[arm, seed]['candidate']['constants']
        for key, value in expected_constants[arm].items():
            assert math.isclose(actual[key], value, abs_tol=1e-12), (arm, seed, key)

print(f'Validated {len(runs)} paired runs, {len(SEEDS)} seeds per condition, and contract {CONTRACT}.')


In [ ]:
def fmt(value, digits=4):
    return f'{value:.{digits}f}'


def table(headers, rows):
    def cell(value):
        return str(value).replace('|', r'\|').replace('\n', ' ')
    line = '| ' + ' | '.join(map(cell, headers)) + ' |'
    sep = '| ' + ' | '.join(['---'] * len(headers)) + ' |'
    return '\n'.join([line, sep] + ['| ' + ' | '.join(map(cell, row)) + ' |' for row in rows])


def paired_ci(values):
    """Two-sided 95% Student t interval for the 11 matched seed differences (df=10)."""
    if len(values) != 11:
        raise ValueError('The prespecified interval requires all 11 seeds')
    mean = statistics.mean(values)
    half_width = 2.2281388519649385 * statistics.stdev(values) / math.sqrt(11)
    return mean, mean - half_width, mean + half_width


def candidate(arm, seed, metric):
    return runs[arm, seed]['candidate'][metric]


def baseline(seed, metric):
    return runs[3, seed]['baseline'][metric]


def paired_deltas(arm, metric):
    return [candidate(arm, seed, metric) - baseline(seed, metric) for seed in SEEDS]

print('Table and paired-seed summary helpers ready.')


## 2. Primary score and matched-seed contrasts

The 95% intervals use the 11 **seed-level paired differences**, with a two-sided Student t multiplier for 10 degrees of freedom. They describe variation across these seeds; no formal hypothesis tests or multiple-comparison correction were specified in the paper. The paper reports Conditions 3 and 4 in its main score table. The Condition 6 score below comes from the same saved evaluation contract.


In [ ]:
score_rows = []
for arm in ARMS:
    diffs = paired_deltas(arm, 'BeliefUpdateScore')
    mean, low, high = paired_ci(diffs)
    score_rows.append({
        'condition': arm,
        'baseline_mean': statistics.mean(baseline(s, 'BeliefUpdateScore') for s in SEEDS),
        'candidate_mean': statistics.mean(candidate(arm, s, 'BeliefUpdateScore') for s in SEEDS),
        'paired_gain': mean, 'ci_low': low, 'ci_high': high,
        'seeds_passing': sum(selections[arm, s]['keep'] for s in SEEDS),
    })
display(Markdown(table(
    ['Condition', 'Baseline mean', 'Candidate mean', 'Paired gain', '95% paired CI', 'Seeds passing'],
    [[r['condition'], fmt(r['baseline_mean']), fmt(r['candidate_mean']),
      f"{r['paired_gain']:+.4f}", f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]",
      f"{r['seeds_passing']}/11"] for r in score_rows])))

contrast_rows = []
for left, right in ((4, 3), (6, 4), (6, 3)):
    differences = [candidate(left, s, 'BeliefUpdateScore') - candidate(right, s, 'BeliefUpdateScore')
                   for s in SEEDS]
    mean, low, high = paired_ci(differences)
    contrast_rows.append({'contrast': f'{left} − {right}', 'mean_difference': mean,
                          'ci_low': low, 'ci_high': high})
display(Markdown('**Direct candidate contrasts, paired by seed**\n\n' + table(
    ['Condition contrast', 'Mean score difference', '95% paired CI'],
    [[r['contrast'], f"{r['mean_difference']:+.4f}",
      f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]"] for r in contrast_rows])))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
colors = {3: '#5277A5', 4: '#D98740', 6: '#528A6B'}
for arm in ARMS:
    ax.plot(SEEDS, paired_deltas(arm, 'BeliefUpdateScore'), marker='o', linewidth=1.8,
            label=f'Condition {arm}', color=colors[arm])
ax.axhline(0, color='#555555', linewidth=1)
ax.axhline(0.02, color='#777777', linestyle='--', linewidth=1,
           label='Minimum score gain for a per-seed pass')
ax.set(title='BeliefUpdateScore: candidate minus matched baseline',
       xlabel='Training seed', ylabel='Score difference', xticks=SEEDS)
ax.legend(ncol=2, frameon=False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## 3. Belief, action, and safety tradeoffs

These are **means of per-seed metrics**, with the baseline counted once per seed. A lower ignored-evidence, perseveration, or unsafe-commit rate is better. Higher entropy indicates more diffuse beliefs and is not by itself a success measure. Revision accuracy and novel-reversal accuracy capture different slices of belief updating.


In [ ]:
metric_labels = {
    'revision_accuracy': 'Revision accuracy',
    'novel_reversal_revision_accuracy': 'Novel-reversal accuracy',
    'ignored_evidence_rate': 'Ignored evidence',
    'moved_but_uncertain_rate': 'Moved but uncertain',
    'entropy_mean': 'Mean belief entropy',
    'perseveration': 'Perseveration',
    'unsafe_commit_rate': 'Unsafe commitments',
}
metric_rows = []
for metric, label in metric_labels.items():
    row = {'metric': metric, 'label': label,
           'baseline': statistics.mean(baseline(s, metric) for s in SEEDS)}
    for arm in ARMS:
        row[f'condition_{arm}'] = statistics.mean(candidate(arm, s, metric) for s in SEEDS)
    metric_rows.append(row)
display(Markdown(table(
    ['Measure', 'Baseline', 'Condition 3', 'Condition 4', 'Condition 6'],
    [[r['label'], fmt(r['baseline']), fmt(r['condition_3']),
      fmt(r['condition_4']), fmt(r['condition_6'])] for r in metric_rows])))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
profiles = [
    (axes[0], ['revision_accuracy', 'novel_reversal_revision_accuracy',
               'ignored_evidence_rate', 'moved_but_uncertain_rate', 'entropy_mean'], 'Belief measures'),
    (axes[1], ['perseveration', 'unsafe_commit_rate'], 'Action and safety measures'),
]
lookup = {r['metric']: r for r in metric_rows}
for ax, keys, title in profiles:
    x = list(range(len(keys)))
    for idx, (name, field, color) in enumerate((
        ('Baseline', 'baseline', '#999999'),
        ('Condition 3', 'condition_3', colors[3]),
        ('Condition 4', 'condition_4', colors[4]),
        ('Condition 6', 'condition_6', colors[6]),
    )):
        ax.bar([v + (idx - 1.5) * 0.20 for v in x], [lookup[k][field] for k in keys],
               width=0.19, color=color, label=name)
    ax.set(title=title, ylim=(0, 0.62), xticks=x,
           xticklabels=[metric_labels[k].replace(' ', '\n') for k in keys])
    ax.grid(axis='y', alpha=0.2)
axes[0].legend(ncol=2, frameon=False, loc='upper right')
plt.tight_layout()
plt.show()


## 4. Why individual seeds passed or failed

A per-seed pass requires `BeliefUpdateScore` gain of at least 0.02 **and** increases of no more than 0.05 in unsafe commitment, perseveration, and ignored evidence. The decision is read from the saved selection report and independently checked here. A higher score can still fail a safety or behavior gate.


In [ ]:
gate_rows = []
for arm in ARMS:
    for seed in SEEDS:
        selection = selections[arm, seed]
        changes = {key: candidate(arm, seed, key) - baseline(seed, key)
                   for key in ('unsafe_commit_rate', 'perseveration', 'ignored_evidence_rate')}
        gain = candidate(arm, seed, 'BeliefUpdateScore') - baseline(seed, 'BeliefUpdateScore')
        independently_passed = gain >= 0.02 - 1e-12 and all(v <= 0.05 + 1e-12 for v in changes.values())
        assert independently_passed == selection['keep'], (arm, seed)
        failures = [name for name, passed in (
            ('score gain', gain >= 0.02 - 1e-12),
            ('unsafe commit', changes['unsafe_commit_rate'] <= 0.05 + 1e-12),
            ('perseveration', changes['perseveration'] <= 0.05 + 1e-12),
            ('ignored evidence', changes['ignored_evidence_rate'] <= 0.05 + 1e-12),
        ) if not passed]
        gate_rows.append({'condition': arm, 'seed': seed, 'score_gain': gain,
                          'decision': 'Pass' if selection['keep'] else 'Fail',
                          'failed_gates': ', '.join(failures) or '—'})
display(Markdown(table(
    ['Condition', 'Seed', 'Score gain', 'Decision', 'Failed gate(s)'],
    [[r['condition'], r['seed'], f"{r['score_gain']:+.4f}", r['decision'], r['failed_gates']]
     for r in gate_rows])))


## 5. Task outcomes reconstructed from the saved traces

The saved traces predate the `episode_return` and terminal-outcome fields now emitted by `eval.py`. For each evaluation episode, this cell reconstructs reward using the experiment's fixed schedule: −0.01 per step, an additional −0.02 for probing, and terminal rewards of +0.40/−0.40 for yield in a closed/open world, +0.55/−0.45 for proceed in an open/closed world, and +1/−1 for commit in an open/closed world. A terminal action is appropriate when it is proceed or commit in an open world, or yield in a closed world. The result is validated against the paper's 165-episode descriptive totals.

Episode counts are descriptive; uncertainty is calculated at the **11-seed** level. Mean reward is an evaluation outcome and is distinct from the `learning_curve.csv` training return.


In [ ]:
def episode_outcome(scenario):
    steps = scenario['steps']
    if not steps:
        raise ValueError(f"No steps in {scenario['name']}")
    rewards = []
    for index, step in enumerate(steps):
        action, world = step['action'], step['world']
        if action not in ('wait', 'probe', 'yield', 'proceed', 'commit') or world not in ('open', 'closed'):
            raise ValueError((scenario['name'], action, world))
        reward = -0.01
        if action == 'probe':
            reward -= 0.02
        elif action == 'yield':
            reward += 0.40 if world == 'closed' else -0.40
        elif action == 'proceed':
            reward += 0.55 if world == 'open' else -0.45
        elif action == 'commit':
            reward += 1.0 if world == 'open' else -1.0
        if action in ('yield', 'proceed', 'commit') and index != len(steps) - 1:
            raise ValueError(f"Terminal action before final trace step: {scenario['name']}")
        if 'reward' in step:
            assert math.isclose(step['reward'], reward, abs_tol=1e-10)
        rewards.append(reward)
    action, world = steps[-1]['action'], steps[-1]['world']
    terminal = action in ('yield', 'proceed', 'commit')
    appropriate = terminal and ((world == 'open' and action in ('proceed', 'commit')) or
                                (world == 'closed' and action == 'yield'))
    outcome = 'Appropriate' if appropriate else 'Inappropriate' if terminal else 'No terminal action'
    total = sum(rewards)
    if 'episode_return' in scenario:
        assert math.isclose(scenario['episode_return'], total, abs_tol=1e-10)
    return total, outcome

utility = {}
for label, arm, role in [('Baseline', 3, 'baseline')] + [(f'Condition {a}', a, 'candidate') for a in ARMS]:
    episodes = []
    per_seed_mean = []
    for seed in SEEDS:
        seed_episodes = [episode_outcome(s) for s in traces[arm, seed, role]]
        episodes.extend(seed_episodes)
        per_seed_mean.append(statistics.mean(value for value, _ in seed_episodes))
    counts = Counter(outcome for _, outcome in episodes)
    utility[label] = {'mean_reward': statistics.mean(per_seed_mean),
                      'per_seed_mean': per_seed_mean, 'counts': counts, 'n': len(episodes)}
    assert len(episodes) == 165

# Published descriptive totals provide an independent check on the trace reconstruction.
expected = {
    'Baseline': (-0.0046, 39, 28, 98),
    'Condition 3': (-0.0515, 45, 42, 78),
    'Condition 4': (-0.0341, 31, 24, 110),
    'Condition 6': (-0.0428, 60, 57, 48),
}
for label, (reward, good, bad, no_terminal) in expected.items():
    row = utility[label]
    assert round(row['mean_reward'], 4) == reward, label
    assert tuple(row['counts'][key] for key in
                 ('Appropriate', 'Inappropriate', 'No terminal action')) == (good, bad, no_terminal), label
print('Trace-derived task outcomes match the paper.')

display(Markdown(table(
    ['Group', 'Mean reward', 'Appropriate', 'Inappropriate', 'No terminal action', 'Unsafe-commit rate'],
    [[label, fmt(utility[label]['mean_reward']),
      f"{utility[label]['counts']['Appropriate']} ({utility[label]['counts']['Appropriate']/165:.1%})",
      f"{utility[label]['counts']['Inappropriate']} ({utility[label]['counts']['Inappropriate']/165:.1%})",
      f"{utility[label]['counts']['No terminal action']} ({utility[label]['counts']['No terminal action']/165:.1%})",
      f"{statistics.mean((baseline(s, 'unsafe_commit_rate') if label == 'Baseline' else candidate(int(label[-1]), s, 'unsafe_commit_rate')) for s in SEEDS):.1%}"]
     for label in ('Baseline', 'Condition 3', 'Condition 4', 'Condition 6')])))


In [ ]:
labels = ('Baseline', 'Condition 3', 'Condition 4', 'Condition 6')
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
stack_bottom = [0] * len(labels)
for category, color in (('Appropriate', '#579274'), ('Inappropriate', '#C36B65'),
                        ('No terminal action', '#A9ADB1')):
    values = [utility[label]['counts'][category] / 165 for label in labels]
    axes[0].bar(labels, values, bottom=stack_bottom, color=color, label=category)
    stack_bottom = [a + b for a, b in zip(stack_bottom, values)]
axes[0].set(title='Terminal outcome composition', ylabel='Share of evaluation episodes', ylim=(0, 1))
axes[0].legend(frameon=False, fontsize=9)
axes[0].tick_params(axis='x', rotation=20)

mean_rewards = [utility[label]['mean_reward'] for label in labels]
axes[1].bar(labels, mean_rewards, color=['#999999', colors[3], colors[4], colors[6]])
axes[1].axhline(0, color='#555555', linewidth=1)
axes[1].set(title='Mean total reward per evaluation episode', ylabel='Mean reward')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

reward_rows = []
base_rewards = utility['Baseline']['per_seed_mean']
for arm in ARMS:
    values = utility[f'Condition {arm}']['per_seed_mean']
    mean, low, high = paired_ci([a - b for a, b in zip(values, base_rewards)])
    reward_rows.append((arm, f'{mean:+.4f}', f'[{low:+.4f}, {high:+.4f}]'))
display(Markdown('**Mean reward change versus matched baseline (seed-level 95% intervals)**\n\n' +
                 table(['Condition', 'Paired reward change', '95% paired CI'], reward_rows)))


## 6. A belief-to-action example in Condition 4

The paper notes that Condition 4's seeds 23 and 31 failed the perseveration gate. In the novel closed-to-open reversals, the belief that the world was open sometimes exceeded 0.7 while the agent continued to wait. The table below retrieves those moments directly from the saved traces. It is a case study, not another independent sample.


In [ ]:
case_rows = []
for seed in (23, 31):
    for scenario in traces[4, seed, 'candidate']:
        if not scenario['name'].startswith('novel_reversal'):
            continue
        qualifying = [step for step in scenario['steps']
                      if step['world'] == 'open' and step['belief_after'][0] >= 0.7
                      and step['action'] == 'wait']
        for step in qualifying:
            case_rows.append((seed, scenario['name'], step['step'],
                              fmt(step['belief_after'][0], 3), step['action']))
display(Markdown(table(['Seed', 'Scenario', 'Step', 'P(open) after cue', 'Action'], case_rows)))
assert case_rows, 'Expected novel-reversal examples were not found'


## 7. Downloadable summary tables and interpretation

The following CSVs contain the displayed aggregate results, direct score contrasts, per-seed gate outcomes, and task outcomes. In Modal, download them from the Files panel. For persistence after the kernel stops, save them to an attached Volume.

**Reading the results:** Condition 4 has the largest mean belief-score gain in these logs. Condition 3 reduces perseveration but weakens novel-reversal accuracy. Condition 6 combines low perseveration with more terminal decisions, including more inappropriate decisions and a higher unsafe-commit rate. The score, safety, terminal outcomes, and reward should therefore be read together. The experiment is a small computational test suite; these findings do not establish a clinical or pharmacological effect.


In [ ]:
OUTPUT_DIR = Path.cwd() / 'conditions_3_4_6_outputs'  # Change to a mounted /mnt/... Volume to persist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def write_csv(name, rows):
    path = OUTPUT_DIR / name
    with path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return path

files = [
    write_csv('score_summary.csv', score_rows),
    write_csv('score_contrasts.csv', contrast_rows),
    write_csv('metric_profiles.csv', metric_rows),
    write_csv('seed_gate_decisions.csv', gate_rows),
    write_csv('task_outcomes.csv', [
        {'group': label, 'mean_episode_reward': utility[label]['mean_reward'],
         'appropriate_endings': utility[label]['counts']['Appropriate'],
         'inappropriate_endings': utility[label]['counts']['Inappropriate'],
         'no_terminal_endings': utility[label]['counts']['No terminal action'],
         'evaluation_episodes': utility[label]['n']}
        for label in ('Baseline', 'Condition 3', 'Condition 4', 'Condition 6')]),
]
print('Wrote:')
for path in files:
    print(' ', path)
